# Тема 1. Что такое видео: кадры, FPS, сжатие, цвет

Сегодня разберём видео «по косточкам»: из чего оно состоит, почему его можно сжать в 200 раз, как сделать свой видео-продакшн и почему компьютеру удобнее искать цвет в HSV, а не в RGB. В конце посмотрим, как видео выглядит в PyTorch.

**Как работать с ноутбуком**
- Запускайте ячейки по порядку сверху вниз: `Shift + Enter`.
- Перед ячейками с пометкой **Предскажи** сначала напишите ответ в ячейке-ответе, потом запускайте. Ошибиться не страшно — страшно не подумать.
- **Задание** — место, где вы меняете код сами. Обычно нужно поменять 1–3 числа или строки.
- Если всё сломалось: меню «Среда выполнения → Перезапустить сеанс» и запустить всё заново.

**Что сдаём (2 балла LXP):** видео со своей подписью, логотипом и временем с тенью (часть 5) + ответы в ячейках «Мой ответ».

In [ ]:
# Ячейка настройки. Просто запустите её, разбираться в ней не нужно.
import sys, importlib.util, subprocess
for module, package in {'cv2': 'opencv-python-headless', 'torch': 'torch', 'PIL': 'pillow'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

import os, time, shutil, pathlib, zipfile
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, Javascript, display
import torch
import torch.nn.functional as F

OUT = pathlib.Path('results')
OUT.mkdir(exist_ok=True)
plt.rcParams['animation.embed_limit'] = 60  # МБ, чтобы анимации помещались в ноутбук
IN_COLAB = 'google.colab' in sys.modules
print('OpenCV', cv2.__version__, '| PyTorch', torch.__version__, '| Colab:', IN_COLAB)


def show(frame, title='', size=5):
    # Показать BGR-кадр OpenCV. matplotlib ждёт RGB, поэтому переставляем каналы.
    plt.figure(figsize=(size, size * frame.shape[0] / frame.shape[1]))
    if frame.ndim == 2:
        plt.imshow(frame, cmap='gray', vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()


def show_row(frames, titles, size=3.2):
    # Несколько картинок в ряд.
    fig, axes = plt.subplots(1, len(frames), figsize=(size * len(frames), size * 0.8))
    for ax, frame, title in zip(np.atleast_1d(axes), frames, titles):
        if frame.ndim == 2:
            ax.imshow(frame, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()


def play(frames, fps=30, max_frames=90, width=320):
    # Плеер прямо в ноутбуке: кнопки, ползунок, можно смотреть по одному кадру.
    step = max(1, int(np.ceil(len(frames) / max_frames)))
    shown = frames[::step]
    h, w = shown[0].shape[:2]
    scale = min(1.0, width / w)
    shown = [cv2.resize(f, (int(w * scale), int(h * scale))) for f in shown]
    fig = plt.figure(figsize=(4, 4 * shown[0].shape[0] / shown[0].shape[1]))
    plt.axis('off')
    to_rgb = (lambda f: f) if shown[0].ndim == 2 else (lambda f: cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    im = plt.imshow(to_rgb(shown[0]), cmap='gray', vmin=0, vmax=255)
    anim = animation.FuncAnimation(fig, lambda i: im.set_data(to_rgb(shown[i])),
                                   frames=len(shown), interval=1000 * step / fps)
    plt.close(fig)
    display(HTML(anim.to_jshtml(default_mode='loop')))


def save_video(frames, path, fps=30, codec='mp4v'):
    # Записать список BGR-кадров в файл.
    path = pathlib.Path(path)
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), fps, (w, h))  # (ширина, высота)!
    if not writer.isOpened():
        raise RuntimeError(f'Кодек {codec} не открылся')
    for frame in frames:
        writer.write(frame)
    writer.release()
    return path


def read_video(path, max_frames=300, width=None):
    # Прочитать видео в список кадров. Возвращает (кадры, fps).
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f'Не удалось открыть {path}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not (1 <= fps <= 240):
        print(f'FPS в файле = {fps}, это неправдоподобно. Считаем, что FPS = 30.')
        fps = 30.0
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if width is not None and frame.shape[1] != width:
            frame = cv2.resize(frame, (width, round(frame.shape[0] * width / frame.shape[1])))
        frames.append(frame)
    cap.release()
    if not frames:
        raise RuntimeError('В видео нет кадров')
    return frames, fps


def video_info(path):
    cap = cv2.VideoCapture(str(path))
    info = {
        'ширина': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'высота': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'FPS': round(cap.get(cv2.CAP_PROP_FPS), 2),
        'кадров': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    if info['FPS'] > 0:
        info['длительность, с'] = round(info['кадров'] / info['FPS'], 2)
    info['размер файла, КБ'] = round(os.path.getsize(path) / 1024, 1)
    return info

## Часть 1. Кадр — это таблица чисел

Цветной кадр в OpenCV — массив `высота × ширина × 3`. В каждой клетке три числа от 0 до 255.

**Важно:** OpenCV хранит каналы в порядке **B, G, R** (синий, зелёный, красный), а не RGB.

**Предскажи.** В следующей ячейке мы заполним весь кадр числами `(255, 0, 0)`. Какого цвета будет картинка?

**Мой ответ:** ...

In [ ]:
frame = np.zeros((240, 320, 3), dtype=np.uint8)  # 240 строк (высота), 320 столбцов (ширина), 3 канала
frame[:, :] = (255, 0, 0)
print('Форма кадра:', frame.shape)
print('Пиксель в строке 10, столбце 20:', frame[10, 20])
show(frame, 'Кадр, залитый (255, 0, 0)')

Синий! Первое число — синий канал. Запомните: в OpenCV **(B, G, R)**.

Теперь нарисуем на кадре мяч. `cv2.circle(кадр, (x, y), радиус, цвет, -1)` — `x` идёт вправо, `y` вниз, `-1` значит «залить».

**Задание 1.1.** Сделайте фон тёмно-серым, а мяч — жёлтым. Подсказка: жёлтый = красный + зелёный.

In [ ]:
BACKGROUND = (40, 40, 40)   # B, G, R
BALL_COLOR = (255, 0, 0)    # поменяйте на жёлтый

frame = np.zeros((240, 320, 3), dtype=np.uint8)
frame[:, :] = BACKGROUND
cv2.circle(frame, (160, 120), 30, BALL_COLOR, -1)
print('Цвет в центре мяча:', frame[120, 160], '| цвет в углу:', frame[0, 0])
show(frame, 'Мой первый кадр')

## Часть 2. Видео — это стопка кадров

Сделаем видео сами: нарисуем 60 кадров, на каждом мяч чуть правее. Номер кадра напишем прямо на кадре.

In [ ]:
def make_ball_video(n_frames=60, w=320, h=240):
    frames = []
    for i in range(n_frames):
        frame = np.full((h, w, 3), 40, dtype=np.uint8)
        x = 30 + i * (w - 60) // (n_frames - 1)          # мяч едет слева направо
        y = 120 - int(70 * np.sin(np.pi * i / (n_frames - 1)))  # и подпрыгивает
        cv2.circle(frame, (x, y), 20, (0, 200, 255), -1)
        cv2.putText(frame, f'frame {i}', (10, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        frames.append(frame)
    return frames

ball = make_ball_video()
video = np.stack(ball)   # склеиваем список кадров в один 4-мерный массив
print('Форма видео (кадры, высота, ширина, каналы):', video.shape)
print('Всего чисел:', video.size, '→', round(video.nbytes / 1024**2, 1), 'МБ в памяти')
show_row([ball[0], ball[30], ball[59]], ['кадр 0', 'кадр 30', 'кадр 59'])
play(ball, fps=30)

Нажмите на паузу и пролистайте кадры стрелками в плеере: видео — это просто картинки по очереди.

### FPS: откуда у кадров берётся время

Сами кадры не знают своего времени. Время задаёт FPS при записи файла:  **длительность = кадров / FPS**.

**Предскажи.** Сохраним одни и те же 60 кадров с FPS = 30 и с FPS = 10. Сколько секунд будет длиться каждое видео?

**Мой ответ:** ...

In [ ]:
for fps in [30, 10]:
    path = save_video(ball, OUT / f'ball_{fps}fps.mp4', fps=fps)
    print(f'FPS = {fps}:', video_info(path))

Кадры одинаковые, а длительность разная. Если скачать оба файла и открыть в плеере, второй будет в 3 раза медленнее.

**Задание 2.1.** Часто, чтобы ускорить обработку, берут не все кадры, а каждый второй: `ball[::2]`.
1. Сохраните `ball[::2]` с FPS = 30. Какая длительность? Быстрее или медленнее стало движение?
2. Какой FPS нужно поставить, чтобы видео из каждого второго кадра длилось столько же, сколько исходное?

In [ ]:
half = ball[::2]
print('Кадров было', len(ball), '→ стало', len(half))
FPS_FOR_HALF = 30  # подберите FPS, чтобы длительность стала 2 секунды

path = save_video(half, OUT / 'ball_half.mp4', fps=FPS_FOR_HALF)
print(video_info(path))

**Мой ответ:** ...

### Сколько у нас есть времени на кадр

Если видео идёт с камеры вживую, на обработку кадра есть `1000 / FPS` миллисекунд. Не успели — изображение начинает отставать.

In [ ]:
for fps in [24, 30, 60, 120]:
    print(f'{fps:>3} FPS → {1000 / fps:5.1f} мс на кадр')

## Часть 3. Сколько весит видео и зачем сжатие

Посчитаем вес несжатого видео Full HD.

In [ ]:
frame_bytes = 1080 * 1920 * 3          # высота × ширина × 3 канала, по 1 байту
MB, GB, TB = 10**6, 10**9, 10**12
print('Один кадр Full HD:', round(frame_bytes / MB, 1), 'МБ')
print('Секунда на 30 FPS:', round(frame_bytes * 30 / MB), 'МБ')
print('Минута:', round(frame_bytes * 30 * 60 / GB, 1), 'ГБ')
print('Фильм на 2 часа:', round(frame_bytes * 30 * 7200 / TB, 2), 'ТБ')

Настоящий фильм весит несколько гигабайт. Значит, кодек сжимает в сотни раз. Сравним два кодека на нашем мяче:
- `MJPG` сжимает каждый кадр отдельно, как JPEG-картинку;
- `mp4v` хранит только **разницу** между соседними кадрами.

In [ ]:
def compare_codecs(frames, name):
    raw = np.stack(frames).nbytes
    row = {'видео': name, 'без сжатия, КБ': round(raw / 1024)}
    for codec, ext in [('MJPG', 'avi'), ('mp4v', 'mp4')]:
        path = save_video(frames, OUT / f'{name}_{codec}.{ext}', fps=30, codec=codec)
        size = os.path.getsize(path)
        row[f'{codec}, КБ'] = round(size / 1024)
        row[f'{codec}, сжатие в'] = round(raw / size)
    return row

print(compare_codecs(ball, 'ball'))

Почему разница между кадрами так помогает? Посмотрим, сколько пикселей реально меняется между соседними кадрами.

In [ ]:
diff = cv2.absdiff(ball[10], ball[11])            # модуль разницы для каждого пикселя
changed = (diff.max(axis=2) > 0).mean() * 100
print(f'Между кадрами 10 и 11 изменилось {changed:.1f}% пикселей')
show_row([ball[10], ball[11], diff * 4], ['кадр 10', 'кадр 11', 'разница (×4 для наглядности)'])

**Предскажи.** Сделаем «видео» из случайного шума: каждый кадр не похож на предыдущий. Во сколько раз его сожмёт `mp4v` по сравнению с мячом: так же, лучше или хуже?

**Мой ответ:** ...

In [ ]:
rng = np.random.default_rng(0)
noise = [rng.integers(0, 256, (240, 320, 3), dtype=np.uint8) for _ in range(60)]
print(compare_codecs(noise, 'noise'))
show(noise[0], 'Кадр шума', size=3)

Шум почти не сжимается: кодеку нечего переиспользовать. В жизни так ведут себя вода, листва, конфетти, снег — на таких сценах видео в плохом интернете «рассыпается на квадратики».

**Задание 3.1.** Что сожмётся сильнее: мяч на чёрном фоне, который **стоит на месте**, или мяч, который **двигается**? Проверьте: сделайте 60 одинаковых кадров `[ball[0]] * 60` и сравните.

In [ ]:
# ваш код здесь: используйте compare_codecs

**Мой ответ:** ...

## Часть 4. Своё видео: демо, файл или веб-камера

Выберите источник в поле `SOURCE` и запустите ячейку:
- `demo` — готовое демо-видео: три цветных объекта, а на середине ролика набегает «облако» и становится темнее. Оно пригодится в части 6;
- `upload` — загрузить свой файл (до 10–15 секунд);
- `webcam` — записать 5 секунд с веб-камеры. Браузер спросит разрешение на камеру.

Мы берём не больше 150 кадров и уменьшаем их до ширины 320 пикселей, чтобы всё работало быстро.

In [ ]:
def make_demo_video(n_frames=90, w=320, h=240, seed=0):
    # Демо-сцена: красный мяч, зелёный мяч и синий квадрат. После 40-го кадра «облако» затемняет сцену.
    rng = np.random.default_rng(seed)
    gradient = np.linspace(150, 200, w).astype(np.uint8)
    background = np.dstack([np.tile(gradient, (h, 1))] * 3)
    background[h * 2 // 3:] = (100, 115, 110)  # «пол»
    frames = []
    for i in range(n_frames):
        frame = background.copy()
        cv2.circle(frame, (20 + i * 3, 70), 18, (40, 40, 220), -1)                  # красный
        cv2.circle(frame, (160 + int(110 * np.sin(i / 9)), 130), 16, (60, 200, 60), -1)  # зелёный
        x, y = 280 - i * 3, 150 - abs(int(60 * np.sin(i / 7)))
        cv2.rectangle(frame, (x, y), (x + 35, y + 35), (200, 90, 30), -1)          # синий
        light = 1.0 if i < 40 else max(0.45, 1.0 - (i - 40) * 0.14)
        frame = frame.astype(np.float32) * light + rng.normal(0, 2, frame.shape)  # свет + шум камеры
        frames.append(np.clip(frame, 0, 255).astype(np.uint8))
    return frames


def record_webcam(seconds=5, filename='webcam.webm'):
    # Запись с веб-камеры через браузер (работает только в Colab).
    from google.colab.output import eval_js
    from base64 import b64decode
    display(Javascript('''
    async function recordVideo(ms) {
      const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
      const video = document.createElement('video');
      video.srcObject = stream; video.muted = true; video.width = 320;
      document.body.appendChild(video);
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      const recorder = new MediaRecorder(stream, {mimeType: 'video/webm'});
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      const stopped = new Promise(resolve => recorder.onstop = resolve);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, ms));
      recorder.stop(); await stopped;
      stream.getTracks().forEach(t => t.stop()); video.remove();
      const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
      let binary = '';
      for (let i = 0; i < bytes.length; i += 0x8000) binary += String.fromCharCode.apply(null, bytes.subarray(i, i + 0x8000));
      return btoa(binary);
    }
    '''))
    print(f'Запись {seconds} с... Смотрите в камеру и подвигайтесь!')
    data = eval_js(f'recordVideo({int(seconds * 1000)})')
    pathlib.Path(filename).write_bytes(b64decode(data))
    return filename


def to_constant_fps_mp4(path, fps=30):
    # Записи браузера и телефонов часто без FPS или с «плавающим» FPS. ffmpeg (есть в Colab) это чинит.
    if shutil.which('ffmpeg') is None:
        return path
    fixed = str(pathlib.Path(path).with_suffix('')) + '_30fps.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path), '-r', str(fps), '-an', fixed], check=True)
    return fixed


SOURCE = 'demo'  # @param ['demo', 'upload', 'webcam']

if SOURCE == 'webcam':
    my_path = to_constant_fps_mp4(record_webcam(5))
elif SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    my_path = to_constant_fps_mp4(next(iter(uploaded)))
else:
    my_path = save_video(make_demo_video(), OUT / 'demo.mp4', fps=30)

print('Свойства файла:', video_info(my_path))
frames, fps = read_video(my_path, max_frames=150, width=320)
H, W = frames[0].shape[:2]
print(f'Прочитано кадров: {len(frames)}, размер кадра {W}×{H}, FPS = {fps}')
play(frames, fps)

**Вопрос.** Посмотрите на свойства файла и на прочитанные кадры. Совпадает ли «длительность» из свойств с `кадров / FPS`? Если вы снимали на телефон или веб-камеру, могли ли эти числа разойтись и почему?

**Мой ответ:** ...

## Часть 5. Свой видео-продакшн

Добавим на видео три вещи, как в видеоредакторе:
1. **Время с тенью** — задание LXP «Вывести время с тенью».
2. **Подпись** на русском языке.
3. **Логотип** с прозрачностью и анимацией — задание LXP «Свой видео-продакшн».

### 5.1 Время с тенью

Почему текст рисуют два раза? Белый текст не виден на светлом фоне. Тень — тот же текст тёмным цветом со сдвигом на 2 пикселя. Он делает надпись читаемой на любом фоне.

**Задание 5.1 (LXP, 1 балл).** Поменяйте шрифт на другой (список ниже), цвет текста и цвет тени. Попробуйте сдвиг тени 1, 2 и 4 пикселя — какой читается лучше?

In [ ]:
FONTS = {
    'SIMPLEX': cv2.FONT_HERSHEY_SIMPLEX, 'PLAIN': cv2.FONT_HERSHEY_PLAIN,
    'DUPLEX': cv2.FONT_HERSHEY_DUPLEX, 'COMPLEX': cv2.FONT_HERSHEY_COMPLEX,
    'TRIPLEX': cv2.FONT_HERSHEY_TRIPLEX, 'SCRIPT': cv2.FONT_HERSHEY_SCRIPT_SIMPLEX,
}
FONT = 'SIMPLEX'                # @param ['SIMPLEX', 'PLAIN', 'DUPLEX', 'COMPLEX', 'TRIPLEX', 'SCRIPT']
TEXT_COLOR = (255, 255, 255)    # B, G, R
SHADOW_COLOR = (0, 0, 0)
SHADOW_SHIFT = 2


def put_text_with_shadow(frame, text, x, y, scale=0.6):
    font = FONTS[FONT]
    cv2.putText(frame, text, (x + SHADOW_SHIFT, y + SHADOW_SHIFT), font, scale, SHADOW_COLOR, 2, cv2.LINE_AA)
    cv2.putText(frame, text, (x, y), font, scale, TEXT_COLOR, 1, cv2.LINE_AA)
    return frame


def time_text(i, fps):
    seconds = i / fps    # время кадра = номер / FPS
    return f'{int(seconds // 60):02d}:{seconds % 60:05.2f}'


example = put_text_with_shadow(frames[0].copy(), time_text(45, fps), 10, 25)
show(example, 'Время с тенью')

### 5.2 Подпись на русском

Шрифты OpenCV не знают кириллицу (вместо букв будут `???`). Поэтому подпись рисуем через библиотеку PIL нормальным шрифтом. Шрифт DejaVu Sans лежит внутри matplotlib, поэтому он точно есть.

In [ ]:
FONT_FILE = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf', 'DejaVuSans-Bold.ttf')

CAPTION = 'Иван Иванов · ML'  # @param {type:"string"}
CAPTION_SIZE = 16             # @param {type:"integer"}


def put_caption(frame, text, x, y, size=CAPTION_SIZE, color=(255, 255, 255), shadow=(0, 0, 0)):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))   # OpenCV (BGR) → PIL (RGB)
    draw = ImageDraw.Draw(image)
    font = ImageFont.truetype(FONT_FILE, size)
    draw.text((x + 2, y + 2), text, font=font, fill=shadow[::-1])     # [::-1]: BGR → RGB
    draw.text((x, y), text, font=font, fill=color[::-1])
    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)            # обратно в BGR


show(put_caption(frames[0], CAPTION, 10, H - 30), 'Подпись через PIL')

### 5.3 Логотип с прозрачностью

У логотипа 4 канала: B, G, R и **альфа** — непрозрачность от 0 до 1. Наложение — формула для каждого пикселя:

$$\text{результат} = \alpha \cdot \text{логотип} + (1 - \alpha) \cdot \text{кадр}$$

Логотип нарисуем кодом. Можно загрузить свой PNG с прозрачным фоном (поле `LOGO_FILE`).

In [ ]:
# Свой логотип: загрузите PNG в панель «Файлы» слева и впишите имя, например 'logo.png'
LOGO_FILE = ''  # @param {type:"string"}
LOGO_SIZE = 56


def make_logo(size=LOGO_SIZE):
    logo = np.zeros((size, size, 4), dtype=np.uint8)               # B, G, R, альфа
    cv2.circle(logo, (size // 2, size // 2), size // 2 - 2, (230, 120, 20, 255), -1)
    cv2.circle(logo, (size // 2, size // 2), size // 2 - 2, (255, 255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(logo, 'ML', (size // 2 - 15, size // 2 + 8), cv2.FONT_HERSHEY_DUPLEX, 0.8,
                (255, 255, 255, 255), 2, cv2.LINE_AA)
    return logo


def load_logo():
    if LOGO_FILE and os.path.exists(LOGO_FILE):
        logo = cv2.imread(LOGO_FILE, cv2.IMREAD_UNCHANGED)
        if logo.shape[2] == 3:                                        # нет альфы — делаем непрозрачной
            logo = np.dstack([logo, np.full(logo.shape[:2], 255, np.uint8)])
        return cv2.resize(logo, (LOGO_SIZE, LOGO_SIZE * logo.shape[0] // logo.shape[1]))
    return make_logo()


def overlay(frame, logo, x, y, opacity=1.0):
    # Наложить логотип в точку (x, y). Части логотипа за краем кадра обрезаются.
    result = frame.copy()
    h, w = logo.shape[:2]
    x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, frame.shape[1]), min(y + h, frame.shape[0])
    if x0 >= x1 or y0 >= y1:
        return result
    part = logo[y0 - y:y1 - y, x0 - x:x1 - x]
    alpha = part[:, :, 3:4].astype(np.float32) / 255 * opacity
    region = result[y0:y1, x0:x1].astype(np.float32)
    result[y0:y1, x0:x1] = (alpha * part[:, :, :3] + (1 - alpha) * region).astype(np.uint8)
    return result


logo = load_logo()
show_row([overlay(frames[0], logo, W - LOGO_SIZE - 8, 8, op) for op in [1.0, 0.5, 0.2]],
         ['α = 1', 'α = 0.5', 'α = 0.2'])

### 5.4 Монтажный план

В Colab нет нажатий клавиш, поэтому включение элементов задаём временем. Логика та же, что у кнопки: на каждом кадре проверяем «включено ли сейчас» и решаем, что рисовать.

Анимация — это когда положение или прозрачность зависят от времени `t`. Здесь логотип плавно проявляется за 1 секунду, а потом слегка «дышит».

**Задание 5.2 (LXP, 1 балл).**
1. Впишите свою подпись в `CAPTION` (ячейка 5.2).
2. Поменяйте монтажный план: когда появляются подпись и логотип.
3. Измените анимацию логотипа: например, пусть он въезжает слева или вращается вокруг угла кадра. Подсказка: меняйте `x` и `y` в зависимости от `t`.

In [ ]:
CAPTION_FROM = 1.0   # секунда, с которой видна подпись
LOGO_FROM = 1.5      # секунда, с которой появляется логотип


def produce(frame, i, fps):
    t = i / fps
    result = frame.copy()
    if t >= LOGO_FROM:
        fade = min(1.0, t - LOGO_FROM)                 # проявление за 1 секунду: 0 → 1
        pulse = 0.85 + 0.15 * np.sin(2 * np.pi * t)    # «дыхание»
        result = overlay(result, logo, W - LOGO_SIZE - 8, 8, opacity=fade * pulse)
    if t >= CAPTION_FROM:
        result = put_caption(result, CAPTION, 10, H - 30)
    result = put_text_with_shadow(result, time_text(i, fps), 10, 25)
    return result


production = [produce(f, i, fps) for i, f in enumerate(frames)]
final_path = save_video(production, OUT / 'my_production.mp4', fps=fps)
print('Сохранено:', video_info(final_path))
play(production, fps)

Файл `results/my_production.mp4` — это то, что сдаётся в LXP. Скачать: панель «Файлы» слева → папка `results` → правой кнопкой → «Скачать». Или запустите последнюю ячейку ноутбука, она соберёт всё в архив.

**Хотите с клавишами `c` и `l`, как в задании LXP?** Запустите на своём компьютере скрипт `video_production_local.py`. Он работает с веб-камерой вживую.

## Часть 6. Цвет: RGB против HSV

Для этой части лучше всего подходит демо-видео: на нём на середине «набегает облако». Если вы загрузили своё видео, следующая ячейка всё равно возьмёт демо.

### 6.1 Каналы

Разложим кадр на три канала. Каждый канал — чёрно-белая картинка: светлое место = много этого цвета.

**Предскажи.** На каком из трёх каналов красный мяч будет самым светлым?

**Мой ответ:** ...

In [ ]:
demo = make_demo_video()
bright, dark = demo[5], demo[80]      # до облака и под облаком
b, g, r = cv2.split(bright)
show_row([bright, b, g, r], ['кадр', 'канал B', 'канал G', 'канал R'])

### 6.2 Эксперимент с облаком

Возьмём пиксель в центре зелёного мяча на светлом и на тёмном кадре и сравним числа в BGR и в HSV.

В OpenCV: **H** (оттенок) от 0 до 179, **S** (насыщенность) и **V** (яркость) от 0 до 255.

**Предскажи.** Свет стал примерно в 2 раза слабее. Какие из шести чисел (B, G, R, H, S, V) сильно изменятся, а какие почти нет?

**Мой ответ:** ...

In [ ]:
def green_center(i):
    return 160 + int(110 * np.sin(i / 9)), 130      # так мы двигали зелёный мяч в make_demo_video


for name, i in [('светло', 5), ('облако', 80)]:
    x, y = green_center(i)
    bgr = demo[i][y, x]
    hsv = cv2.cvtColor(demo[i], cv2.COLOR_BGR2HSV)[y, x]
    print(f'{name:>7}:  BGR = {str(bgr):<16} HSV = {hsv}')

hsv_bright = cv2.cvtColor(bright, cv2.COLOR_BGR2HSV)
show_row([bright, *cv2.split(hsv_bright)], ['кадр', 'H — оттенок', 'S — насыщенность', 'V — яркость'])

Облако изменило все числа BGR и яркость V, а **оттенок H почти не изменился**. Зелёный мяч остался зелёным, просто стал темнее.

Посмотрите на картинку канала H ещё раз:
- **Серый фон — сплошной шум.** У серого цвета нет оттенка, и H там случайный. Поэтому в маске всегда ставят минимальную насыщенность `S_MIN`: она отсекает серое.
- **Красный мяч в крапинку** (чёрное с белыми точками). Красный лежит на границе круга: часть пикселей получила H ≈ 0, часть H ≈ 179. Это пригодится в задании 6.2.

### 6.3 Маска: ищем зелёный мяч

`cv2.inRange(картинка, нижняя_граница, верхняя_граница)` возвращает маску: белый (255) там, где все три числа пикселя внутри границ, чёрный (0) — где нет.

Сравним два правила:
- **в BGR:** «G больше 150, B и R меньше 120»;
- **в HSV:** «оттенок от 45 до 75, насыщенность и яркость не слишком маленькие».

Пороги подобраны на **светлом** кадре. Проверяем на **тёмном**.

In [ ]:
def mask_bgr(frame):
    return cv2.inRange(frame, (0, 150, 0), (120, 255, 120))


H_LOW = 45   # @param {type:"integer"}
H_HIGH = 75  # @param {type:"integer"}
S_MIN = 80   # @param {type:"integer"}
V_MIN = 40   # @param {type:"integer"}


def mask_hsv(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, (H_LOW, S_MIN, V_MIN), (H_HIGH, 255, 255))


for name, frame in [('светло', bright), ('облако', dark)]:
    mb, mh = mask_bgr(frame), mask_hsv(frame)
    print(f'{name:>7}: найдено пикселей  BGR = {np.count_nonzero(mb):4d}   HSV = {np.count_nonzero(mh):4d}')
show_row([bright, mask_bgr(bright), mask_hsv(bright)], ['светло', 'маска BGR', 'маска HSV'])
show_row([dark, mask_bgr(dark), mask_hsv(dark)], ['облако', 'маска BGR', 'маска HSV'])

Маска BGR в тени потеряла мяч, маска HSV — нет.

Теперь применим маску ко всему видео и обведём найденное рамкой. Это «детекция объектов без нейросетей», подробно разберём её на следующей паре.

In [ ]:
def detect(frame, mask_fn, color):
    result = frame.copy()
    contours, _ = cv2.findContours(mask_fn(frame), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        if cv2.contourArea(contour) > 50:            # мелкие пятна шума пропускаем
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(result, (x, y), (x + w, y + h), color, 2)
    return result


tracked = [np.hstack([detect(f, mask_bgr, (0, 0, 255)), detect(f, mask_hsv, (0, 255, 0))]) for f in demo]
print('Слева правило BGR (красная рамка), справа правило HSV (зелёная рамка)')
play(tracked, 30, width=640)

**Задание 6.1.** Найдите **синий квадрат**. В OpenCV синий оттенок ≈ 100–130. Поменяйте `H_LOW`, `H_HIGH` и перезапустите ячейки маски и детекции.

**Задание 6.2 (со звёздочкой).** Найдите **красный мяч**. Сначала попробуйте один диапазон, например 0–10. Затем посмотрите, какой H у красного мяча на самом деле: `cv2.cvtColor(bright, cv2.COLOR_BGR2HSV)[70, 35]`. Красный лежит на **обоих концах** круга (около 0 и около 179), поэтому нужны две маски и их объединение: `cv2.bitwise_or(маска1, маска2)`.

In [ ]:
print('HSV в центре красного мяча:', cv2.cvtColor(bright, cv2.COLOR_BGR2HSV)[70, 35])

def mask_red(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    low = cv2.inRange(hsv, (0, 80, 40), (10, 255, 255))
    # допишите вторую маску для оттенков около 179 и объедините их
    return low

show_row([bright, mask_red(bright), dark, mask_red(dark)], ['светло', 'маска', 'облако', 'маска'])

### 6.4 Серый кадр — это взвешенная сумма

Перевод в серый: `Y = 0.299·R + 0.587·G + 0.114·B`. Зелёный весит больше всех: глаз к нему чувствительнее всего. Так же видеокодеки отделяют яркость от цвета (YUV): яркость хранят точно, а цвет — в 4 раза грубее, и глаз этого почти не замечает.

In [ ]:
gray_cv = cv2.cvtColor(bright, cv2.COLOR_BGR2GRAY)
B, G, R = [c.astype(np.float32) for c in cv2.split(bright)]
gray_formula = np.round(0.299 * R + 0.587 * G + 0.114 * B).astype(np.uint8)
print('Максимальная разница между OpenCV и формулой:', int(np.abs(gray_cv.astype(int) - gray_formula).max()))
show_row([bright, gray_cv], ['цвет', 'серый'])

## Часть 7. Видео в PyTorch

В PyTorch видео — это **тензор**. Порядок осей другой, чем в OpenCV: каналы идут перед высотой и шириной.

| | Форма | Тип |
|---|---|---|
| OpenCV / numpy | `(T, H, W, C)`, каналы BGR | `uint8`, 0–255 |
| PyTorch | `(T, C, H, W)`, каналы RGB | `float32`, 0–1 |

`T` — время (кадры), `C` — каналы, `H` — высота, `W` — ширина.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

x = torch.from_numpy(np.stack(demo))    # (T, H, W, C), uint8, BGR
print('Из numpy:      ', tuple(x.shape), x.dtype)
x = x.permute(0, 3, 1, 2)               # (T, C, H, W)
x = x[:, [2, 1, 0]]                     # BGR → RGB: переставили каналы
x = x.float().div(255).to(device)       # 0..255 → 0..1
print('Для PyTorch:   ', tuple(x.shape), x.dtype, '| устройство:', device)
print('Память: uint8 =', round(np.stack(demo).nbytes / 1024**2, 1), 'МБ, float32 =',
      round(x.element_size() * x.nelement() / 1024**2, 1), 'МБ')

**Вопрос.** Почему `float32` занимает в 4 раза больше памяти? Сколько займёт 10 секунд Full HD на 30 FPS в `float32`?

**Мой ответ:** ...

### 7.1 Одна строка — все кадры

Переведём **все 90 кадров** в серый по формуле из части 6 одной строкой, без цикла.

In [ ]:
weights = torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)  # веса для R, G, B
gray = (x * weights).sum(dim=1)        # сумма по оси каналов → (T, H, W)
print('Серое видео:', tuple(gray.shape))

### 7.2 Детектор движения из разницы кадров

Вычтем из каждого кадра предыдущий. Там, где ничего не двигалось, разница около нуля. Так работает «детекция движения» в простых камерах наблюдения.

**Предскажи.** На демо-видео после 40-го кадра набегает облако. Что покажет детектор движения в это время?

**Мой ответ:** ...

In [ ]:
motion = (gray[1:] - gray[:-1]).abs()            # (T-1, H, W): кадр t+1 минус кадр t
moving = motion > 0.05                           # порог: маленькие изменения — это шум камеры
moving_share = moving.float().mean(dim=(1, 2)).cpu() * 100

plt.figure(figsize=(8, 2.5))
plt.plot(moving_share)
plt.axvspan(39, 44, color='gray', alpha=0.25, label='облако')
plt.xlabel('номер кадра'); plt.ylabel('% «движущихся» пикселей'); plt.legend(); plt.show()

to_img = lambda t: (t.cpu().numpy() * 255).astype(np.uint8)
show_row([to_img(moving[20].float()), to_img(moving[41].float())], ['движение, кадр 20', 'движение, кадр 41 (облако)'])

На кадрах с облаком «движутся» почти все пиксели. Детектор не отличает изменение света от движения — это ложная тревога. Реальная проблема уличных камер: ночью они «видят движение» от фар и теней.

### 7.3 Фон из медианы: стираем движущиеся объекты

Для каждого пикселя возьмём медиану по времени. В большинстве кадров в этой точке был фон, поэтому движущиеся объекты исчезают. Так убирают туристов со снимков достопримечательностей.

In [ ]:
before_cloud = x[:40]                                   # берём кадры до облака
background = before_cloud.median(dim=0).values          # медиана по оси времени → (C, H, W)

rgb_to_bgr_img = lambda t: cv2.cvtColor((t.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
foreground = ((x[20] - background).abs().max(dim=0).values > 0.15).float()
show_row([demo[20], rgb_to_bgr_img(background), to_img(foreground)], ['кадр 20', 'фон (медиана)', 'что не фон'])

**Задание 7.1.** Посчитайте медиану по **всем** кадрам `x`, а не только до облака. Что стало с фоном? Почему?

In [ ]:
# ваш код здесь

**Мой ответ:** ...

### 7.4 Свёртка на все кадры сразу: так смотрит нейросеть

Фильтр Собеля находит вертикальные края: он отвечает «насколько справа светлее, чем слева». Применим его через `F.conv2d` — ту же операцию, из которой состоят свёрточные нейросети.

Хитрость: ось времени `T` становится размером батча. Один и тот же фильтр применяется к каждому кадру. Нейросеть для видео делает ровно это, только веса фильтров не задаются руками, а подбираются при обучении.

In [ ]:
sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]], device=device).view(1, 1, 3, 3)   # (выход, вход, 3, 3)

edges = F.conv2d(gray.unsqueeze(1), sobel_x, padding=1).abs()  # (T, 1, H, W) — все кадры за один вызов
print('Вход:', tuple(gray.unsqueeze(1).shape), '→ выход:', tuple(edges.shape))
edges_img = [to_img((e[0] / edges.max()).clamp(0, 1)) for e in edges]
play(edges_img, 30)

### 7.5 Сколько это стоит по времени

Сравним: цикл OpenCV по кадрам против одного вызова PyTorch для всего видео. И сравним с бюджетом `1000 / FPS` мс на кадр.

In [ ]:
start = time.perf_counter()
for frame in demo:
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.Sobel(g, cv2.CV_32F, 1, 0)
opencv_ms = (time.perf_counter() - start) * 1000 / len(demo)

start = time.perf_counter()
gray_t = (x * weights).sum(dim=1, keepdim=True)
edges_t = F.conv2d(gray_t, sobel_x, padding=1)
if device == 'cuda':
    torch.cuda.synchronize()     # видеокарта работает асинхронно, ждём окончания
torch_ms = (time.perf_counter() - start) * 1000 / len(demo)

print(f'OpenCV, цикл по кадрам: {opencv_ms:.3f} мс на кадр')
print(f'PyTorch ({device}), все кадры сразу: {torch_ms:.3f} мс на кадр')
print(f'Бюджет при 30 FPS: {1000 / 30:.1f} мс на кадр')

Обе операции укладываются в бюджет с огромным запасом: это простые фильтры.

На процессоре OpenCV обычно быстрее: его функции написаны на C++ и вылизаны именно под такие операции. Выигрыш PyTorch появляется на видеокарте, где тысячи ядер считают все кадры параллельно, и в нейросетях, которых в OpenCV просто нет. Настоящая нейросеть — это сотни таких свёрток подряд, и там вопрос «успеваем ли за 33 мс» становится главным. Поэтому модели для видео уменьшают кадры и работают на видеокарте.

**Вопрос.** Если в Colab включена видеокарта (меню «Среда выполнения → Сменить среду → T4 GPU»), PyTorch на первом запуске может оказаться *медленнее*. Перезапустите ячейку — что изменилось? Как думаете, почему?

**Мой ответ:** ...

## Итог

Заполните коротко, по одному предложению:

| | Мой ответ |
|---|---|
| Видео для компьютера — это... | |
| FPS влияет на... | |
| Видео хорошо сжимается, потому что... | |
| HSV лучше RGB для поиска цвета, потому что... | |
| В PyTorch кадры видео становятся... | |
| Самое неожиданное сегодня: | |

Запустите ячейку ниже: она соберёт все результаты в архив `practice_01.zip`.

In [ ]:
archive = pathlib.Path('practice_01.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for file in OUT.rglob('*'):
        if file.is_file():
            z.write(file, file.relative_to(OUT))
print('Архив:', archive.resolve(), '|', round(archive.stat().st_size / 1024), 'КБ')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))